# 2. Structure of the HDF5 output file

PlatoSim produces three output files:

* an **HDF5** file which is the subject of this notebook
* a **log** file which contains all info, warning, error, and debugging messages
* a copy of the **YAML** input file

We will assume in this notebook that you run PlatoSim and inspect the HDF5 output file from a python environment. The HDF5 file is located in the directory specified in `sim.outputDir`.

<img src="../../figures/PlatoSimStructureOfHDF5.png">

Shown in the figure above the first level shows the group `InputParameters` contains a copy of the configuration parameters from the YAML file. Every HDF5 output file has the Git version of the software saved should you need it.

The second level shows all the groups that contain image data for every exposure. These are the `Images`, `SmearingMaps`, `BiasMaps`, and `ThroughputMaps`.

The third level shows all the groups that contain single frame image data. These are the `PSF`, `BackgroundMap`, `FlatField`, and the `CTI`. Notice that the latter contains a map for each CTI trap specy that is included in the used model (hence 4 species for the Short model).

The fourth level shows all the groups that contains a mix of stellar parameters, payload parameters, and coordinates. The group `StarCatalog` contains the sky coordinates, the pixel coordinates, and the focal plane coordinates of all the stars that were detected during any exposure. The starIDs map the ID from the input starCatalog that is supplied with the configuration. 

The exact output structure of the groups `StarPositions`, `PointLikeGhostPositions`, `ExtendedGhostPositions`, and `Cosmics` depends on the YAML input entry called `GoupByExposure` when controlling what needs to be saved. By default every information is saved in folders per Exposure, however, if this parameter is False the information is saved per StarID. For long baseline simulations (more than 100,000 exposures), we recommend to use the latter option since this can significantly lower the computational resources (time and memory). Note that the information about the cosmic rays are never saved per star, however, it is saved into subfolder of 1000 exposures each.

In this tutorial we show how to inspect and extract information from the HDF5 output file produced by PlatoSim. We present how to use `h5py`, the in-build `h5` function, but we will place the effort on explaining how to fully utilise the in-build `SimFile` class.

Note: apart from the libraries show in this tutorial, it is also possible to investigate HDF(5) files with [PyTables](https://www.pytables.org/index.html), however, this packages is known to course problems with other packages used in PlatoSim and hence we avoid it for our dependencies.

### Setup notebook

In [ ]:
# Reload code outside notebook
%load_ext autoreload
%autoreload 2

# Configure figures in notebook
%matplotlib notebook

### Imports

In [ ]:
import os
import numpy as np
import matplotlib.cm as cm
import matplotlib.pyplot as plt

# PlatoSim
from platosim.simulation   import Simulation
from platosim.simfile      import SimFile
from platosim.matplotlibrc import setup_notebook
setup_notebook()

### Run a default simulation for the tutorial

In [ ]:
# Define inputs and outputs
outputDir      = os.getcwd()
outputFileName = "output_example1"

# Generate simulation object and control the output
sim = Simulation(outputFileName, outputDir=outputDir)

# Point-like ghost included by default, include also extended ghosts
sim["Camera/IncludeExtendedGhosts"] = True

# Write PSFs to the output file
sim["ControlHDF5Content/WriteHighResolutionPSF"] = True
sim["ControlHDF5Content/WriteSubPixelImages"]    = True

# Change sky background to pixel dependent
sim["Sky/SkyBackground/UseConstantSkyBackground"] = 'no'
sim["Sky/SkyBackground/BackgroundValue"]          = -1

# Group per exposure or per star
# NOTE that grouping per star is faster for long duration simulations
sim["ControlHDF5Content/GroupByExposure"] = True

# Select location of subfield on the CCD
sim["SubField/ZeroPointColumn"] = 1000
sim["SubField/ZeroPointRow"]    = 1000

# Run PlatoSim (and allow to overwrite output file)
sim.run(removeOutputFile=True);

---
## 2.1 - Inspect with [h5py](https://docs.h5py.org/en/stable/)
---

We here show a minimal example of using [h5py](https://docs.h5py.org/en/stable/) to inspect your HDF5 output file. E.g. let's fetch the first exposure simulated:

In [ ]:
import h5py
h5file = h5py.File(outputFileName + '.hdf5', 'r')

In [ ]:
# Get an image
im = h5file['Images/image0000000'][:]
im

---
## 2.2 - Inspect with h5
---

PlatoSim has an build-in functionality to use `h5` to print the structure and fetch the information from the HDF5 output file:

In [ ]:
from platosim.h5 import h5get, h5ls

### Function: h5ls

The function **h5ls** takes an HDF5 file object or a HDF5 group as a mandatory argument and shows the complete structure of the HDF5 file or group. Each level is indicated by the following type acronyms, and for attributes their value is shown.

[G] Group <br/>
[D] Dataset <br/>
[a] Attribute

Print the entire HDF5 file - equivalent to specifying only the root group `h5ls(h5file['/'])`:

In [ ]:
h5ls(h5file)

In [ ]:
# Example of a Dataset
h5ls(h5file['BiasMapsLeft'])

In [ ]:
# Example of attributes
h5ls(h5file['InputParameters/CCD/Gain'])

### Function: h5get

With **h5get** you can get data out of the HDF5 file into numpy arrays or python variables. This function takes two mandatory arguments, the HDF5 file object (or group) and the 'path into the variable or dataset'.

When you specify the full path, only that variable is returned as shown in the following two commands. When you specify a partial path or just the name of the final dataset/attribute, the **h5get** function looks for all possible matches and returns an array with their values. This is illustrated further with 'ReadoutNoise'.

In [ ]:
pos = h5get(h5file, "/InputParameters/CCD/Position", verbose=False)
print(f"Type and value of Position: {type(pos)}, {pos}")

In [ ]:
noise = h5get(h5file, "/InputParameters/CCD/ReadoutNoise", verbose=False)
print(f"Type and value of ReadoutNoise: {type(noise)}, {noise}")

In [ ]:
h5get(h5file, "ReadoutNoise")

In [ ]:
cols = h5get(h5file, "InputParameters/CCD/NumColumns", verbose=False)
print ("Type and value of NumColumns: {}, {}".format(type(cols), cols))

In [ ]:
dec = h5get(h5file, "ACS/PlatformDec", verbose=False)
ra  = h5get(h5file, "ACS/PlatformRA", verbose=False)
print ("Type and shape of RA : {}, {}".format(type(ra), ra.shape))
print ("Type and shape of Dec: {}, {}".format(type(dec), dec.shape))

In [ ]:
im = h5get(h5file,"Images/image0000000", verbose=False)
im

---
## 2.3 - Inspect with SimFile
---

In the previous tutorials we already touched upon how you can use the PlatoSim `SimFile` class to retrieve information from the HDF5 output file. Here we dive in a bit deeper and show most of its functionalites. 

A few notes:
- All functions automatically access which camera (*Normal* or *Fast*) that has been used in the simulation.
- Many of the following functionalities it is possible to add the time column of the camera cadence by parsing the flag `getTime=True`. Look out for when this is applicable in the following. 
- By default the help functions of `SimFile` return numpy arrays wherever applicable, however, to make the data handling more user friendly it is also possible to parse the flag `df=True` to return a Pandas data frame. 

First let's get the HDF5 file with:

In [ ]:
f = SimFile(outputFileName + ".hdf5")

`reload()`: If you accidentially overwrites/mess up some parameters, you can always reload the HDF5 output again using: 

In [ ]:
f.reload()

For the sake of this tutorial we will only inspect the first exposure, hence:

In [ ]:
imageNr = 0

### Input Parameters

`getInputParameter()`: We here give a minimal example to show how to inspect one of the input parameters that was used for the simulation. The group name and parameter name are exactly the same as in the YAML file. E.g. fetch the `CycleTime`:

In [ ]:
f.getInputParameter("ObservingParameters", "CycleTime")

`getExposureTime()`: Get the exposure time:

In [ ]:
t_exp = f.getExposureTime()
t_exp

`getReadoutTime()`: Get the readout time before the next exposure starts and the readout time during the next exposure:

In [ ]:
t_out_before, t_out_after = f.getReadoutTime()
t_out_before, t_out_after

``getTime()``: Get time column of all exposures - Note that this is a Pandas data frame:

In [ ]:
time = f.getTime(df=True)
time

### Pointing, Jitter, and Drift

Below we show how to fetch information from the `Platform` and `Telescope` group in the HDF5 file. Both of groups share the same data structure of corresponding columns, hence, below we simple show the functionalities for the platform. Simple replace "Platform" with "Telescope" in the respective function name to fetch information about the telescope.

`getPlatformInfo()`: Get all the columns stored in the platform group:

In [ ]:
f.getPlatformInfo(df=True).head()

 `getPlatformYawPitchRoll()`: Get the platform rotation angles (yaw, pitch, roll):

In [ ]:
df = f.getPlatformYawPitchRoll(getTime=True, df=True)

In [ ]:
plt.figure(figsize=(7,4))
exp = np.arange(1,11,1)
plt.plot(df.time, df.yaw,   'o--', label="Yaw")
plt.plot(df.time, df.pitch, 'o--', label="Pitch")
plt.plot(df.time, df.roll,  'o--', label="Roll")
plt.xlabel("Time [s]")
plt.ylabel("Amplitude [arcsec]")
plt.tight_layout()
plt.legend();

`getPlatformPointingCoordinates()`: Get the platform jitter in either equatorial coordinates (RA, Dec):

In [ ]:
alpha, delta = f.getPlatformPointingCoordinates()

# Convert to relative pointing coordinates [deg -> arcsec]
alphaMean = (alpha-np.mean(alpha)) * 3600
deltaMean = (delta-np.mean(delta)) * 3600

In [ ]:
plt.figure(figsize=(6,6))
plt.plot(alphaMean, deltaMean, 'o--')
plt.xlabel("Relative RA [arcsec]")
plt.ylabel("Relative Dec [aecsec]")
plt.ticklabel_format(useOffset=False) 
plt.tight_layout()
plt.show()

### Ghosts

`getPointLikeGhostCoordinates()`: Get information about the point-like ghosts:

In [ ]:
f.getPointLikeGhostCoordinates(imageNr, df=True).head()

`getExtendedGhostCoordinates()`: Get information about extended ghosts:

In [ ]:
f.getExtendedGhostCoordinates(imageNr, df=True).head()

### Cosmic rays

`getCosmicsInfo()`: You can also retrieve information about the cosmic rays added to each exposure. Since cosmic rays can be added to several image products, note that the `field` parameters can determines from what field the Cosmics should be returned: `SubField`, `BiasMapLeft`, `BiasMapRight` or `SmearingMap`. We here fetch the cosmic ray information for exposure 0 in the star image:

In [ ]:
f.getCosmicsInfo(imageNr, field="SubField", df=True)

`getCosmicsAffectedPixels()`: Get knowledge about the pixels that are affected by cosmic rays in the subfield:

In [ ]:
f.getCosmicsAffectedPixels(imageNr, df=True).head()

### Stellar information

`getStarCatalog()`: Get the catalogue containing the stars that have been detected on a CCD:

In [ ]:
f.getStarCatalog(df=True).head()

`getStarCoordinates()`: Get the stellar coordinates in the CCD focal plane (both in pixel and mm). It is also possible to only get the coordinates of stars within a certain magnitude range `[minMag, maxMag]` as we show in the following for exposre 0:

In [ ]:
df = f.getStarCoordinates(imageNr, minMag=None, maxMag=None, df=True)
df.head()

`getStarPosition()`: Get the stellar pixel coordinates only for a specific target star ID:

In [ ]:
f.getStarPositions(df.ID.iloc[0], getTime=True, df=True)

### Subfield images

`getImage()`: Get the full simulated subfield and the image dimentions and use them as extentions:

In [ ]:
image      = f.getImage(imageNr)
numRows    = f.getInputParameter("SubField", "NumRows")
numColumns = f.getInputParameter("SubField", "NumColumns")

`showMap()`: We additionally provide a small quicklook tool that can be used to plot any pixel map:

In [ ]:
fig, ax = f.showMap(image, figsize=(6,5))
ax.scatter(np.floor(df.col), np.floor(df.row), marker='+', c='r');

Due the large dynamic range in pixel counts, this is unfortunately not well suited for larger subfields. Another options is to use the build-in function `showImage()` to visualize your subfield together with the stellar coordinates:

In [ ]:
fig, ax = f.showImage(imageNr, imgScale="percentile", clip=1,
                      minMag=10.0, maxMag=12.0,
                      figsize=(7,7), fontSize=18, useTitle=False,
                      showStarPositions=True, showStarIDs=False,
                      colorMap="cubehelix", colorBar=True, showGrid=True)

`getImagette()`: It is also possible to fetch a smaller subfield (known as an *imagette*) around a given target star. Here we first find all the star IDs and select a subfield with radius of 3 pixels:

In [ ]:
imagette = f.getImagette(df.ID.iloc[0], imageNr, radius=3)

In [ ]:
f.showMap(imagette, figsize=(6,5));

### Point Spred Function (PSF)

`getPSF()`: The PSF can also be fetched. Above we saved the high resolution PSF to file, but it is also possible to save the diffused PSF. Here we fetch the following:

In [ ]:
psf = f.getPSF("highResPSF")

`showPSF()`: Again there is an in-build function to show the used PSF:

In [ ]:
fig, ax = f.showPSF("highResPSF");

### Remaing pixel maps

`getSmearingMap()`: The smearing map (from a parallel overscan) can be fetched with:

In [ ]:
sm = f.getSmearingMap(imageNr)

In [ ]:
f.showMap(sm, figsize=(8,3));

`getBiasMapLeft()` and `getBiasMapRight()`: Get the bias map (from a serial prescan) for the left and right CCD half, respectively:

In [ ]:
bm_left, bm_right = f.getBiasMapLeft(imageNr), f.getBiasMapRight(imageNr)

In [ ]:
f.showMap(bm_left, figsize=(5,5))
f.showMap(bm_right, figsize=(5,5));

`getSkyBackgroun()`: The sky background map can be fetched with:

In [ ]:
bg = f.getBackground()

In [ ]:
f.showMap(bg, figsize=(6,5), clabel='Photons/s/pixel');

 `getThroughputMap()`: Get the throughput map:

In [ ]:
tm = f.getThroughputMap(imageNr)

In [ ]:
f.showMap(tm, figsize=(6,5), clabel='Normalised counts');

`getPRNU()`: Get the Pixel Response Non-Uniformity (PRNU; which essentially is the flat-field image):

In [ ]:
prnu = f.getPRNU()

In [ ]:
f.showMap(prnu, figsize=(6,5), clabel='Relative counts');